# Part 1: Topic Detection and Summarization of News Articles

Generative AI - Assignment 1
Darrsheni Sapovadia (27PGAI0063)

The BBC News Archive has 2,225 articles spread over five categories. The brief asks for the
first 30, so that is what I work with here. For each article I use LangChain to do three
things: decide the topic, write a short summary, and pull out the key entities. All three
results get added back onto the dataframe at the end.

I am calling Groq for the model. The assignment mentions `llama-3.1-8b-instant` but Groq has
since retired the Llama models, so I used `openai/gpt-oss-120b` instead. It is on the same
free tier and is quick enough for classification and extraction work.

## Setup

In [1]:
import json
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

load_dotenv("../.env")

MODEL = os.getenv("LLM_MODEL", "openai/gpt-oss-120b")

# temperature 0 because these are extraction tasks, not creative writing -
# I want the same answers if I run the notebook again.
#
# reasoning_effort took me a while to work out. This model thinks before it
# answers, and the thinking counts towards max_tokens. On the longer articles it
# spent the whole budget reasoning and handed back an empty string, which is why
# my entity column kept coming out blank. Turning the effort down fixes it and
# makes the run a good deal faster too.
llm = ChatGroq(model=MODEL, temperature=0, max_tokens=800, reasoning_effort="low")

print("Model:", MODEL)

Model: openai/gpt-oss-120b


In [2]:
def ask(chain, article, limit=2500):
    """Run a chain on one article, waiting it out if Groq rate limits us.

    The free tier allows 8,000 tokens a minute and news articles are long, so
    being rate limited here is normal rather than exceptional. Groq puts the
    wait it wants in the error message, so I pull that out and use it instead of
    guessing. If it still fails after six goes I let it raise, because a blank
    string coming back would just turn into an empty cell that I might not spot.
    """
    for attempt in range(6):
        try:
            return chain.invoke({"article": article[:limit]})
        except Exception as error:
            message = str(error)
            limited = "rate" in message.lower() or "429" in message
            if not limited or attempt == 5:
                raise
            suggested = re.search(r"try again in ([\d.]+)s", message)
            time.sleep(float(suggested.group(1)) + 2 if suggested else 20)


def find_json(text):
    """Grab the JSON object out of a reply and ignore anything around it."""
    found = re.search(r"\{.*\}", text, re.S)
    if not found:
        return {}
    try:
        return json.loads(found.group())
    except json.JSONDecodeError:
        return {}

## Step 1: Load the dataset

The file is tab separated rather than comma separated, which caught me out the first time.
I keep the original category column too so I can check the predictions at the end.

In [3]:
news = pd.read_csv("../data/bbc-news-data.csv", sep="\t")

print("Whole dataset:", news.shape)
news.head(3)

Whole dataset: (2225, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...


In [4]:
df = news.head(30).copy().reset_index(drop=True)
df = df.rename(columns={
    "title": "Title",
    "content": "Article_Text",
    "category": "True_Category",
})
df.insert(0, "Article_ID", df.index + 1)
df = df[["Article_ID", "Title", "Article_Text", "True_Category"]]

print("Working set:", df.shape)
df.head()

Working set: (30, 4)


,Article_ID,Title,Article_Text,True_Category
0,1,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,business
1,2,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,business
2,3,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,business
3,4,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,business
4,5,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,business


## Step 2: Topic classification

I give the model the five categories and a few worked examples before the real article. The
examples are mainly there to stop it answering in a full sentence.

The two sentences in the system prompt about companies and the economy were not in my first
version. I added them after looking at what it was getting wrong. The BBC files anything to
do with trade, prices or the economy under business even when it is a government doing it,
and without that steer the model kept calling those articles Politics. That is a perfectly
reasonable reading, it just is not the one the dataset uses, and spelling the rule out fixed
most of those cases.

In [5]:
CATEGORIES = ["Business", "Entertainment", "Politics", "Sport", "Tech"]

topic_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a news desk editor. Put each article into exactly one category: "
     "Business, Entertainment, Politics, Sport or Tech.\n"
     "Anything about companies, markets, jobs, trade, prices or the wider economy is "
     "Business, even when governments, courts or regulators are involved. "
     "Keep Politics for stories that are mainly about elections, parties, or the running "
     "of government itself.\n"
     "Reply with the category name and nothing else."),
    ("human",
     "Analyze the following news article and identify its topic as one of the following "
     "categories: Business, Entertainment, Politics, Sport, or Tech.\n\n"
     "Article: Shares in the airline fell 4% after it warned that fuel costs would wipe "
     "out full-year profits."),
    ("ai", "Business"),
    ("human",
     "Article: The government will lift fuel subsidies next month, a move economists warn "
     "will push inflation higher and squeeze household budgets."),
    ("ai", "Business"),
    ("human",
     "Article: The home secretary faces a backbench rebellion over plans to introduce "
     "compulsory identity cards."),
    ("ai", "Politics"),
    ("human",
     "Article: The striker scored twice in the second half to send his side into the "
     "cup final."),
    ("ai", "Sport"),
    ("human", "Article: {article}"),
])

topic_chain = topic_prompt | llm | StrOutputParser()

In [6]:
def classify_topic(article):
    reply = ask(topic_chain, article, limit=1500)
    # every so often it adds a stray word or a full stop, so match against the list
    for category in CATEGORIES:
        if category.lower() in reply.lower():
            return category
    return "Other"

In [7]:
sample = df.loc[0]

print("Title:", sample["Title"])
print("Actual category:", sample["True_Category"])
print("Predicted topic:", classify_topic(sample["Article_Text"]))

Title: Ad sales boost Time Warner profit
Actual category: business


Predicted topic: Business


## Step 3: Summarization

Two or three sentences, and I tell it to stay with what the article actually says so it does
not start editorialising.

In [8]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You summarise news articles. Cover who, what, when and where. "
     "Stick to what the article says and do not add any opinion of your own."),
    ("human",
     "Summarize the main points of the following news article in 2-3 sentences.\n\n"
     "Article:\n{article}"),
])

summary_chain = summary_prompt | llm | StrOutputParser()


def summarize(article):
    return ask(summary_chain, article).strip()

In [9]:
print("Title:", sample["Title"])
print()
print("Summary:", summarize(sample["Article_Text"]))

Title: Ad sales boost Time Warner profit



Summary: TimeWarner’s quarterly profit jumped 76% to $1.13 billion for the three months to December, helped by higher sales of high‑speed internet, stronger advertising revenue and one‑off gains that offset a dip at Warner Bros and a loss of AOL subscribers. The company’s fourth‑quarter sales rose 2% to $11.1 billion, it now owns 8% of Google, and it is preparing to restate its 2000 and 2003 results after an SEC probe into AOL while projecting about 5% operating‑earnings growth for 2005.


## Step 4: Key entity extraction

I ask for JSON with three groups so the output is easy to parse, then flatten it into one
list per article. Asking for plain text here gave me bullet points and headings that were a
nuisance to clean up.

In [10]:
entity_prompt = ChatPromptTemplate.from_messages([
    ("system", "You pull named entities out of news articles. Reply with JSON and nothing else."),
    ("human",
     "From the article below, list the names of any important people, organisations or "
     "places mentioned.\n"
     'Reply in this exact shape: {{"people": [], "organisations": [], "places": []}}\n'
     "Leave a list empty if there is nothing for it.\n\n"
     "Article:\n{article}"),
])

entity_chain = entity_prompt | llm | StrOutputParser()


def get_entities(article):
    data = find_json(ask(entity_chain, article))
    names = []
    for group in ("people", "organisations", "places"):
        for name in data.get(group, []):
            name = str(name).strip()
            if name and name not in names:
                names.append(name)
    return names

In [11]:
print("Title:", sample["Title"])
print()
print("Key entities:", get_entities(sample["Article_Text"]))

Title: Ad sales boost Time Warner profit



Key entities: ['Richard Parsons', 'TimeWarner', 'Google', 'AOL', 'Warner Bros', 'US Securities Exchange Commission (SEC)', 'Bertelsmann', 'United States']


## Step 5: Run all three tasks over the 30 articles

Three calls per article, so ninety in total. The short sleep keeps us under the
tokens-per-minute limit on the free Groq tier.

In [12]:
topics = []
summaries = []
entities = []

start = time.time()

for position, row in df.iterrows():
    article = row["Article_Text"]

    topics.append(classify_topic(article))
    summaries.append(summarize(article))
    entities.append(get_entities(article))

    print(f"article {position + 1} of {len(df)} done")
    time.sleep(2)

print(f"\nAll finished in {time.time() - start:.0f} seconds")

article 1 of 30 done


article 2 of 30 done


article 3 of 30 done


article 4 of 30 done


article 5 of 30 done


...

article 27 of 30 done


article 28 of 30 done


article 29 of 30 done


article 30 of 30 done



All finished in 385 seconds


In [13]:
df["Detected_Topic"] = topics
df["Summary"] = summaries
df["Key_Entities"] = entities

# the three new columns on their own first
df[["Article_ID", "Detected_Topic", "Summary", "Key_Entities"]].head(10)

,Article_ID,Detected_Topic,Summary,Key_Entities
0,1,Business,TimeWarner’s quarterly profit jumped 76% to $1...,"[Richard Parsons, TimeWarner, Google, AOL, War..."
1,2,Business,"The dollar rose to $1.2871 per euro, its highe...","[Alan Greenspan, Robert Sinche, Federal Reserv..."
2,3,Business,"The owners of Russia’s former oil giant Yukos,...","[Jamie Firestone, Tim Osborne, Mikhail Khodork..."
3,4,Business,British Airways reported a 40 % drop in pre‑ta...,"[Rod Eddington, Mike Powell, Martin Broughton,..."
4,5,Business,Allied Domecq’s London shares rose about 4% af...,"[Allied Domecq, Pernod Ricard, Wall Street Jou..."
5,6,Business,Japan’s revised data show that the economy gre...,"[Heizo Takenaka, Paul Sheard, Lehman Brothers,..."
6,7,Business,"U.S. firms added 146,000 non‑farm jobs in Janu...","[President Bush, Herbert Hoover, Rick Egelton,..."
7,8,Business,India’s finance minister Palaniappan Chidambar...,"[Palaniappan Chidambaram, Gordon Brown, G7, Un..."
8,9,Business,Ethiopia’s crop output rose to 14.27 million t...,"[Henri Josserand, Food and Agriculture Organis..."
9,10,Business,A U.S. appeals court rejected the federal gove...,"[Clinton, US government, Altria Group, RJ Reyn..."


### The full dataframe, original columns and new ones together

In [14]:
pd.set_option("display.max_colwidth", 60)
df

,Article_ID,Title,Article_Text,True_Category,Detected_Topic,Summary,Key_Entities
0,1,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarner jumped 7...,business,Business,TimeWarner’s quarterly profit jumped 76% to $1.13 billio...,"[Richard Parsons, TimeWarner, Google, AOL, Warner Bros, ..."
1,2,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro i...,business,Business,"The dollar rose to $1.2871 per euro, its highest level i...","[Alan Greenspan, Robert Sinche, Federal Reserve, Bank of..."
2,3,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ...,business,Business,"The owners of Russia’s former oil giant Yukos, through M...","[Jamie Firestone, Tim Osborne, Mikhail Khodorkovsky, Yuk..."
3,4,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% d...,business,Business,British Airways reported a 40 % drop in pre‑tax profit f...,"[Rod Eddington, Mike Powell, Martin Broughton, Nick Van ..."
4,5,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have ri...,business,Business,Allied Domecq’s London shares rose about 4% after Wall S...,"[Allied Domecq, Pernod Ricard, Wall Street Journal, Fina..."
5,6,Japan narrowly escapes recession,Japan's economy teetered on the brink of a technical re...,business,Business,Japan’s revised data show that the economy grew only 0.1...,"[Heizo Takenaka, Paul Sheard, Lehman Brothers, Japan, To..."
6,7,Jobs growth still slow in the US,"The US created fewer jobs than expected in January, but...",business,Business,"U.S. firms added 146,000 non‑farm jobs in January, falli...","[President Bush, Herbert Hoover, Rick Egelton, Ken Mayla..."
7,8,India calls for fair trade rules,"India, which attends the G7 meeting of seven leading in...",business,Business,India’s finance minister Palaniappan Chidambaram attende...,"[Palaniappan Chidambaram, Gordon Brown, G7, United Natio..."
8,9,Ethiopia's crop production up 24%,Ethiopia produced 14.27 million tonnes of crops in 2004...,business,Business,Ethiopia’s crop output rose to 14.27 million tonnes in 2...,"[Henri Josserand, Food and Agriculture Organisation, Wor..."
9,10,Court rejects $280bn tobacco case,A US government claim accusing the country's biggest to...,business,Business,A U.S. appeals court rejected the federal government’s $...,"[Clinton, US government, Altria Group, RJ Reynolds Tobac..."


### One row as JSON

Same shape as the example given in the assignment brief.

In [15]:
row = df.loc[0]

print(json.dumps({
    "Article_ID": int(row["Article_ID"]),
    "Title": row["Title"],
    "Article_Text": row["Article_Text"][:200] + "... [excerpt]",
    "Detected_Topic": row["Detected_Topic"],
    "Summary": row["Summary"],
    "Key_Entities": row["Key_Entities"],
}, indent=2))

{
  "Article_ID": 1,
  "Title": "Ad sales boost Time Warner profit",
  "Article_Text": " Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (\u00a3600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google,... [excerpt]",
  "Detected_Topic": "Business",
  "Summary": "TimeWarner\u2019s quarterly profit jumped 76% to $1.13\u202fbillion for the three months to December, helped by higher sales of high\u2011speed internet, stronger advertising revenue and one\u2011off gains that offset a dip at Warner\u202fBros and a loss of AOL subscribers. The company\u2019s fourth\u2011quarter sales rose 2% to $11.1\u202fbillion, it now owns 8% of Google, and it is preparing to restate its 2000 and 2003 results after an SEC probe into AOL while projecting about 5% operating\u2011earnings growth for 2005.",
  "Key_Entities": [
    "Richard Parsons",
    "TimeWarner",
    "Google",
    "AOL",
    "Warner Bros",
    "US S

### Sanity check on the topics

The dataset comes with its own category labels, so I can compare my predictions against
them. Worth flagging one thing though: the file is sorted by category, so the first 30 rows
are all business articles. That makes this a soft test. It tells me whether the classifier
is wrongly firing on the other four categories, but it says nothing about whether it can
actually pick those four out.

In [16]:
check = df[["Article_ID", "Title", "True_Category", "Detected_Topic"]].copy()
check["Match"] = check["True_Category"].str.lower() == check["Detected_Topic"].str.lower()

print(f"Correct: {check['Match'].sum()} out of {len(check)}")
print()

wrong = check[~check["Match"]]
if len(wrong):
    print("Ones it got wrong:")
    print(wrong[["Title", "True_Category", "Detected_Topic"]].to_string(index=False))
else:
    print("Every article matched.")

Correct: 29 out of 30

Ones it got wrong:
                            Title True_Category Detected_Topic
Ask Jeeves tips online ad revival      business           Tech


So I took a second sample from further down the file, two articles from each of the five
categories, and ran the classifier over those as well. That is a fairer test of whether it
can tell the categories apart. The random seed is fixed so this picks the same ten every
time.

In [17]:
mixed = (news.groupby("category", group_keys=False)
             .sample(2, random_state=42)
             .reset_index(drop=True))

results = []
for _, item in mixed.iterrows():
    results.append({
        "Title": item["title"][:42],
        "Actual": item["category"].title(),
        "Predicted": classify_topic(item["content"]),
    })
    time.sleep(2)

spot = pd.DataFrame(results)
spot["Match"] = spot["Actual"].str.lower() == spot["Predicted"].str.lower()

print(f"Correct: {spot['Match'].sum()} out of {len(spot)}")
print()
print(spot.to_string(index=False))

Correct: 10 out of 10

                            Title        Actual     Predicted  Match
 Christmas sales worst since 1981      Business      Business   True
US retail sales surge in December      Business      Business   True
 Fry set for role in Hitchhiker's Entertainment Entertainment   True
 New York rockers top talent poll Entertainment Entertainment   True
  Clarke faces ID cards rebellion      Politics      Politics   True
 Ministers deny care sums 'wrong'      Politics      Politics   True
 Collins named UK Athletics chief         Sport         Sport   True
IAAF to rule on Greek sprint pair         Sport         Sport   True
 Software watching while you work          Tech          Tech   True
  Net fingerprints combat attacks          Tech          Tech   True


## Save the results

In [18]:
out = df.copy()
out["Key_Entities"] = out["Key_Entities"].apply(lambda names: ", ".join(names))
out.to_csv("../outputs/part1_news_results.csv", index=False)

print("Saved", len(out), "rows to outputs/part1_news_results.csv")

Saved 30 rows to outputs/part1_news_results.csv
